In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vectorbt as vbt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Define start and end dates for a 1-year period
end_date = pd.Timestamp.today()
start_date = end_date - pd.DateOffset(years=5)

# Fetch historical data
ocbc = yf.Ticker("O39.SI").history(start=start_date, end=end_date)['Close']
dbs = yf.Ticker("D05.SI").history(start=start_date, end=end_date)['Close']

# Ensure aligned dates
prices = pd.DataFrame({'OCBC': ocbc, 'DBS': dbs}).dropna()
prices

# # Linear regression to find the hedge ratio (beta)
# x = add_constant(prices['OCBC'])  # OCBC as independent variable
# y = prices['DBS']  # DBS as dependent variable
# model = OLS(y, x).fit()
# beta = model.params['OCBC']

# # Calculate the spread
# spread = prices['DBS'] - beta * prices['OCBC']

# # Calculate z-score for the spread
# spread_mean = spread.mean()
# spread_std = spread.std()
# z_score = (spread - spread_mean) / spread_std

# # Generate signals
# long_signal = z_score < -1.5
# short_signal = z_score > 1.5
# exit_signal = z_score.abs() < 0.0

# # Backtest using Vectorbt
# portfolio = vbt.Portfolio.from_signals(
#     close=prices['DBS'],
#     entries=long_signal,
#     exits=exit_signal,
#     short_entries=short_signal,
#     short_exits=exit_signal,
#     size=np.full(len(prices), 10000 / prices['DBS']),  # Fixed position size
#     fees=0.001  # Example fee
# )

# # Plot the spread and z-score
# fig, ax = plt.subplots(2, 1, figsize=(12, 8))
# ax[0].plot(spread, label='Spread (DBS - Beta * OCBC)', color='blue')
# ax[0].axhline(spread_mean, color='red', linestyle='--', label='Mean')
# ax[0].set_title('Spread')
# ax[0].legend()

# ax[1].plot(z_score, label='Z-Score', color='green')
# ax[1].axhline(1.5, color='red', linestyle='--', label='Z = 1.5')
# ax[1].axhline(-1.5, color='red', linestyle='--', label='Z = -1.5')
# ax[1].set_title('Z-Score of Spread')
# ax[1].legend()
# plt.tight_layout()
# plt.show()

# # Display portfolio performance
# portfolio.stats()



,OCBC,DBS
Date,,
2021-06-08 00:00:00+08:00,9.446561,20.645597
2021-06-09 00:00:00+08:00,9.378272,20.432470
2021-06-10 00:00:00+08:00,9.401036,20.549345
2021-06-11 00:00:00+08:00,9.355510,20.439348
2021-06-14 00:00:00+08:00,9.279634,20.391222
...,...,...
2026-06-02 00:00:00+08:00,24.080000,64.669998
2026-06-03 00:00:00+08:00,24.530001,65.050003
2026-06-04 00:00:00+08:00,24.000000,64.139999


In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vectorbt as vbt
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Define start and end dates for a 1-year period
end_date = pd.Timestamp.today()
start_date = end_date - pd.DateOffset(years=1)

# Fetch historical data
ocbc = yf.Ticker("O39.SI").history(start=start_date, end=end_date)['Close']
dbs = yf.Ticker("D05.SI").history(start=start_date, end=end_date)['Close']

# Ensure aligned dates
prices = pd.DataFrame({'OCBC': ocbc, 'DBS': dbs}).dropna()

# Linear regression to find the hedge ratio (beta)
x = add_constant(prices['OCBC'])  # OCBC as independent variable
y = prices['DBS']  # DBS as dependent variable
model = OLS(y, x).fit()
beta = model.params['OCBC']

# Calculate the spread
spread = prices['DBS'] - beta * prices['OCBC']

# Calculate z-score for the spread
spread_mean = spread.mean()
spread_std = spread.std()
z_score = (spread - spread_mean) / spread_std

# Generate signals
long_signal = z_score < -1.5
short_signal = z_score > 1.5
exit_signal = z_score.abs() < 0.0

# Backtest using Vectorbt with both DBS and OCBC positions
portfolio = vbt.Portfolio.from_orders(
    close=prices,
    size=np.where(long_signal[:, None], [10000 / prices['DBS'], -beta * (10000 / prices['OCBC'])],
                  np.where(short_signal[:, None], [-10000 / prices['DBS'], beta * (10000 / prices['OCBC'])], 0)),
    fees=0.001,  # Example fee
    size_type='amount'
)

# Plot the spread and z-score
fig, ax = plt.subplots(2, 1, figsize=(12, 8))
ax[0].plot(spread, label='Spread (DBS - Beta * OCBC)', color='blue')
ax[0].axhline(spread_mean, color='red', linestyle='--', label='Mean')
ax[0].set_title('Spread')
ax[0].legend()

ax[1].plot(z_score, label='Z-Score', color='green')
ax[1].axhline(1.5, color='red', linestyle='--', label='Z = 1.5')
ax[1].axhline(-1.5, color='red', linestyle='--', label='Z = -1.5')
ax[1].set_title('Z-Score of Spread')
ax[1].legend()
plt.tight_layout()
plt.show()

# Display portfolio performance
portfolio.performance()
portfolio.plot().show()


ValueError: Multi-dimensional indexing (e.g. `obj[:, None]`) is no longer supported. Convert to a numpy array before indexing instead.